# **1. Data Retrieval**
**Case Study: Gapminder (Simple)**

Dataset gapminder_simple.csv là phiên bản rút gọn (1000 samples) từ Gapminder. Các cột chính:
*   country: tên quốc gia
*   continent: châu lục
*   year: năm quan sát
*   lifeExp: tuổi thọ trung bình
*   pop: dân số
*   gdpPercap: GDP bình quân đầu người

Mục tiêu: luyện các thao tác retrieval cơ bản (projection, selection, top-N, aggregation,
computed fields) bằng cả Pandas và SQL. Các bạn có thể tải data theo link: https://drive.google.com/file/d/1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs/view

In [1]:
!gdown --id 1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs
To: /content/gapminder_simple.csv
100% 50.2k/50.2k [00:00<00:00, 60.5MB/s]


## **1.1 Import data**

In [2]:
import pandas as pd

df = pd.read_csv('gapminder_simple.csv')
df.head()

,country,year,pop,continent,lifeExp,gdpPercap
0,Myanmar,1962,23634436.0,Asia,45.108,388.000000
1,Ireland,1957,2878220.0,Europe,68.900,5599.077872
2,Jamaica,1977,2156814.0,Americas,70.110,6650.195573
3,Cote d'Ivoire,1987,10761098.0,Africa,54.655,2156.956069
4,Morocco,1997,28529501.0,Africa,67.660,2982.101858


## **1.2 Projection**

In [3]:
use_cols = ["country", "continent", "year", "lifeExp", "gdpPercap"]
df1 = df[use_cols].copy()

df1.head()

# SQL:
# SELECT country , continent , ‘year ‘, lifeExp , gdpPercap
# FROM gapminder_simple
# LIMIT 10;

,country,continent,year,lifeExp,gdpPercap
0,Myanmar,Asia,1962,45.108,388.000000
1,Ireland,Europe,1957,68.900,5599.077872
2,Jamaica,Americas,1977,70.110,6650.195573
3,Cote d'Ivoire,Africa,1987,54.655,2156.956069
4,Morocco,Africa,1997,67.660,2982.101858


## **1.3 Selection**

In [4]:
# Ví dụ: Chỉ lấy Asia, giai đoạn 1990-2007
mask = (df1['continent'] == 'Asia') & (df1['year'] >= 1990) & (df1['year'] <= 2007)
asia_1990_2007 = df.loc[mask,
                        ["country", "continent", "year", "lifeExp", "gdpPercap", "pop"]].copy()

asia_1990_2007.head(10)

# SQL:
# SELECT country , continent , ‘year ‘, lifeExp , gdpPercap , pop
# FROM gapminder_simple
# WHERE continent = ’Asia ’
#   AND ‘year ‘ BETWEEN 1990 AND 2007
# LIMIT 10;

,country,continent,year,lifeExp,gdpPercap,pop
7,Taiwan,Asia,1997,75.250,20206.820980,2.162860e+07
19,Nepal,Asia,2002,61.340,1057.206311,2.587392e+07
28,Mongolia,Asia,1997,63.625,1902.252100,2.494803e+06
40,India,Asia,2007,64.698,2452.210407,1.110396e+09
73,India,Asia,2002,62.879,1746.769454,1.034173e+09
80,Israel,Asia,1992,76.930,18051.522540,4.936550e+06
224,China,Asia,2002,72.028,3119.280896,1.280400e+09
234,Vietnam,Asia,1997,70.672,1385.896769,7.604900e+07
240,Singapore,Asia,2002,78.770,36023.105400,4.197776e+06
247,Myanmar,Asia,1992,59.320,347.000000,4.054654e+07


## **1.4 Ordering & Top-N**

In [5]:
# Lấy 10 quốc gia có gpdPercap cao nhất trong năm 2007
df_2007 = df[df["year"] == 2007].copy()

top10_gdpPercap_2007 = (
    df_2007.sort_values("gdpPercap", ascending=False)
    [["country", "continent", "year", "gdpPercap", "lifeExp"]]
    .head(10)
)

top10_gdpPercap_2007

# SQL:
# SELECT country , continent , ‘year ‘, gdpPercap , lifeExp
# FROM gapminder_simple
# WHERE ‘year ‘ = 2007
# ORDER BY gdpPercap DESC
# LIMIT 10;

,country,continent,year,gdpPercap,lifeExp
483,Ireland,Europe,2007,40675.99635,78.885
354,Switzerland,Europe,2007,37506.41907,81.701
988,Netherlands,Europe,2007,36797.93332,79.762
217,Canada,Americas,2007,36319.23501,80.653
900,Iceland,Europe,2007,36180.78919,81.757
554,Austria,Europe,2007,36126.49270,79.829
643,Denmark,Europe,2007,35278.41874,78.332
986,Australia,Oceania,2007,34435.36744,81.235
133,Finland,Europe,2007,33207.08440,79.313
664,United Kingdom,Europe,2007,33203.26128,79.425


## **1.5 Aggregation (Simple)**

Mục tiêu: tổng hợp theo châu lục trong năm 2007 với 2 chỉ số cơ bản: tuổi thọ trung bình và GDP/người trung bình

In [6]:
tmp = df[df["year"] == 2007].copy()

agg_continent_2007 = (
    tmp.groupby("continent", as_index=False)
    .agg(
        avg_lifeExp=("lifeExp", "mean"),
        avg_gdpPercap=("gdpPercap", "mean"),
    )
    .sort_values("avg_gdpPercap", ascending=False)
)

agg_continent_2007

# SQL:
# SELECT
#   continent ,
#   AVG ( lifeExp ) AS avg_lifeExp ,
#   AVG ( gdpPercap ) AS avg_gdpPercap
# FROM gapminder_simple
# WHERE ‘year ‘ = 2007
# GROUP BY continent
# ORDER BY avg_gdpPercap DESC ;

,continent,avg_lifeExp,avg_gdpPercap
4,Oceania,81.235000,34435.367440
3,Europe,77.739950,24975.915126
1,Americas,75.119923,12113.551428
2,Asia,69.328118,8127.222843
0,Africa,54.895667,3660.861747


## **1.6 Computed fields**

Mục tiêu: tạo GDP xấp xỉ theo công thức GDP = pop * gdpPercap, rồi lấy top theo GDP trong năm 2007.

In [7]:
tmp = df[df["year"] == 2007].copy()
tmp["gdp"] = tmp["pop"] * tmp["gdpPercap"]

top10_gdp_2007 = (
    tmp.sort_values("gdp", ascending=False)
    [["country", "continent", "year", "pop", "gdpPercap", "gdp", "lifeExp"]]
    .head(10)
)

top10_gdp_2007

# SQL
# SELECT
#   country , continent , ‘year ‘,
#   pop , gdpPercap ,
#   (pop * gdpPercap ) AS gdp ,
#   lifeExp
# FROM gapminder_simple
# WHERE ‘year ‘ = 2007
# ORDER BY gdp DESC
# LIMIT 10;

,country,continent,year,pop,gdpPercap,gdp,lifeExp
736,China,Asia,2007,1.318683e+09,4959.114854,6.539501e+12,72.961
854,Japan,Asia,2007,1.274680e+08,31656.068060,4.035135e+12,82.603
40,India,Asia,2007,1.110396e+09,2452.210407,2.722925e+12,64.698
344,Germany,Europe,2007,8.240100e+07,32170.374420,2.650871e+12,79.406
664,United Kingdom,Europe,2007,6.077624e+07,33203.261280,2.017969e+12,79.425
563,Brazil,Americas,2007,1.900106e+08,9065.800825,1.722599e+12,72.390
217,Canada,Americas,2007,3.339014e+07,36319.235010,1.212704e+12,80.653
967,Spain,Europe,2007,4.044819e+07,28821.063700,1.165760e+12,80.941
986,Australia,Oceania,2007,2.043418e+07,34435.367440,7.036584e+11,81.235
988,Netherlands,Europe,2007,1.657061e+07,36797.933320,6.097643e+11,79.762


# **2. Data Aggregation & Calculation**

**Data Aggregation** là gom nhóm dữ liệu theo một hoặc nhiều khóa và tính các thống kê tóm tắt cho từng nhóm. **Calculation** là tạo biến dẫn xuấ (computed filed) để phục vụ phân tích và tổng hợp.

**Đặc điểm chính:**


*   Kết quả thường *giảm số dòng*: mỗi nhóm tương ứng với một dòng tổng hợp

*   Thống kê phổ biến: count, sum, average, min, max
*   **WHERE** lọc trước khi nhóm, **HAVING** lọc sau khi đã tổng hợp


*   Có thể tổng hợp trực tiếp trên biến mới(vd. gdp = pop * gdpPercap).



## **Case study: Bộ dữ liệu Gapminder (Simple)**

gapminder_simple.csv (1000 mẫu) là bản rút gọn là Gapminder, mô tả chỉ số phát triển theo quốc gia và năm.


*   Khóa & phân nhóm: country, year, contient
*   Biến số: lifeExp, pop, pdpPercap; tạo gpd = pop * gdpPercap để tính GPD tổng

Các biến thực hành (keywords): computed field -> overall summary -> groupby(continent) -> groupby(year) -> filter + TopN(2007) -> groupby(continent,year).

Dataset (CSV): https://drive.google.com/file/d/1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs/view





**Bước 0: Import và đọc dataset**

In [8]:
!gdown --id 1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs
To: /content/gapminder_simple.csv
100% 50.2k/50.2k [00:00<00:00, 62.3MB/s]


In [9]:
import pandas as pd

DATA_PATH = '/content/gapminder_simple.csv'
df = pd.read_csv(DATA_PATH)

**Bước 1: Tạo computed field gdp**

In [10]:
df["gdp"] = df["pop"] * df["gdpPercap"]

# SQL
# SELECT pop * gdpPercap AS gdp
# FROM gapminder_simple ;

In [11]:
df

,country,year,pop,continent,lifeExp,gdpPercap,gdp
0,Myanmar,1962,23634436.0,Asia,45.108,388.000000,9.170161e+09
1,Ireland,1957,2878220.0,Europe,68.900,5599.077872,1.611538e+10
2,Jamaica,1977,2156814.0,Americas,70.110,6650.195573,1.434323e+10
3,Cote d'Ivoire,1987,10761098.0,Africa,54.655,2156.956069,2.321122e+10
4,Morocco,1997,28529501.0,Africa,67.660,2982.101858,8.507788e+10
...,...,...,...,...,...,...,...
995,Italy,1967,52667100.0,Europe,71.060,10022.401310,5.278508e+11
996,Indonesia,1992,184816000.0,Asia,62.681,2383.140898,4.404426e+11
997,Netherlands,1967,12596822.0,Europe,73.820,15363.251360,1.935281e+11
998,Rwanda,2007,8860588.0,Africa,46.242,863.088464,7.647471e+09


**Bước 2: Thống kê tổng quan (overall summary)**

In [12]:
overall = df[["lifeExp", "gdpPercap", "pop"]].agg(["count", "mean", "min", "max"])
overall

# SQL
# SELECT
#   COUNT (*) AS n,
#   AVG ( lifeExp ) AS avg_lifeExp ,
#   AVG ( gdpPercap ) AS avg_gdpPercap
# FROM gapminder_simple ;

,lifeExp,gdpPercap,pop
count,1000.00000,1000.000000,1.000000e+03
mean,59.42225,7058.398793,3.285722e+07
min,23.59900,241.165876,6.001100e+04
max,82.60300,80894.883260,1.318683e+09


**Bước 3: Tổng hợp theo châu lục(continent)**
Báo cáo theo coninent: Số quốc gia, dân số tổng, tuổi thọ trung bình

In [13]:
df

,country,year,pop,continent,lifeExp,gdpPercap,gdp
0,Myanmar,1962,23634436.0,Asia,45.108,388.000000,9.170161e+09
1,Ireland,1957,2878220.0,Europe,68.900,5599.077872,1.611538e+10
2,Jamaica,1977,2156814.0,Americas,70.110,6650.195573,1.434323e+10
3,Cote d'Ivoire,1987,10761098.0,Africa,54.655,2156.956069,2.321122e+10
4,Morocco,1997,28529501.0,Africa,67.660,2982.101858,8.507788e+10
...,...,...,...,...,...,...,...
995,Italy,1967,52667100.0,Europe,71.060,10022.401310,5.278508e+11
996,Indonesia,1992,184816000.0,Asia,62.681,2383.140898,4.404426e+11
997,Netherlands,1967,12596822.0,Europe,73.820,15363.251360,1.935281e+11
998,Rwanda,2007,8860588.0,Africa,46.242,863.088464,7.647471e+09


In [16]:
by_continent = (
    df.groupby("continent", as_index=False)
    .agg(
        n_countries=("country", "unique"),
        sum_pop=("pop", "sum"),
        avg_lifeExp=("lifeExp", "mean")
    )
    .sort_values("sum_pop", ascending=False)
)

by_continent

# SELECT
#   continent ,
#    COUNT ( DISTINCT country ) AS n_countries ,
#   SUM(pop) AS sum_pop ,
#   AVG( lifeExp ) AS avg_lifeExp
# FROM gapminder_simple
# GROUP BY continent
# ORDER BY sum_pop DESC ;

,continent,n_countries,sum_pop,avg_lifeExp
2,Asia,"[Myanmar, Vietnam, Taiwan, Jordan, Nepal, Mala...",2.059212e+10,60.539551
1,Americas,"[Jamaica, Costa Rica, Paraguay, Ecuador, Colom...",5.034003e+09,64.491207
3,Europe,"[Ireland, Poland, Albania, Germany, Belgium, G...",3.676182e+09,71.959250
0,Africa,"[Cote d'Ivoire, Morocco, Central African Repub...",3.400048e+09,48.604183
4,Oceania,"[Australia, New Zealand]",1.548706e+08,75.487308


**Bước 4: Xu hướng theo năm**

Mục tiêu: xem xu hướng theo thời gian với AVG(lifeExp) và AVG(gdpPercap)

In [18]:
by_year = (
    df.groupby("year", as_index=False)
    .agg(
        avg_lifeExp=("lifeExp", "mean"),
        avg_gdpPercap=("gdpPercap", "mean")
    )
    .sort_values("year")
)

by_year

# SELECT
# year ,
# AVG ( lifeExp ) AS avg_lifeExp ,
# AVG ( gdpPercap ) AS avg_gdpPercap
# FROM gapminder_simple
# GROUP BY year
# ORDER BY year ;

,year,avg_lifeExp,avg_gdpPercap
0,1952,50.070925,3034.800798
1,1957,51.738824,3662.401893
2,1962,52.859235,3961.913760
3,1967,54.926812,5462.055273
4,1972,57.027808,6015.670342
5,1977,58.763482,7417.728671
6,1982,61.599278,7369.994018
7,1987,63.104689,8471.758004
8,1992,63.281321,8132.786436
9,1997,65.787307,9547.891230


**Bước 5 (đơn giản): Top-N châu lục theo GDP tổng
năm 2007**

Mục tiêu: lọc year=2007, tính sum_gdp theo châu lục, lấy top 5

In [19]:
top5_cont_2007 = (
    df[df["year"] == 2007]
    .groupby("continent", as_index=False)
    .agg(sum_gdp=("gdpPercap", "sum"))
    .sort_values("sum_gdp", ascending=False)
    .head(5)
)

top5_cont_2007

# SELECT
#   continent ,
#   SUM (pop * gdpPercap ) AS sum_gdp
# FROM gapminder_simple
# WHERE year = 2007
# GROUP BY continent
# ORDER BY sum_gdp DESC
# LIMIT 5;

,continent,sum_gdp
3,Europe,499518.302517
1,Americas,157476.168568
2,Asia,138162.788324
0,Africa,87860.681930
4,Oceania,34435.367440


**Bước 6: GDP tổng theo châu lục và năm (continent,
year)**

Mục tiêu: tạo bảng 2 chiều theo continent, year để so sánh theo thời gian trong từng châu lục.

In [20]:
gdp_by_cont_year = (
    df.groupby(["continent", "year"], as_index=False)
    .agg(sum_gdp=("gdp", "sum"))
    .sort_values(["continent", "year"])
)

gdp_by_cont_year

# SELECT
# continent , year ,
# SUM(pop * gdpPercap ) AS sum_gdp
# FROM gapminder_simple
# GROUP BY continent , year
# ORDER BY year , sum_gdp DESC ;

,continent,year,sum_gdp
0,Africa,1952,1.602385e+11
1,Africa,1957,1.997284e+11
2,Africa,1962,3.230962e+11
3,Africa,1967,4.914499e+11
4,Africa,1972,3.360763e+11
5,Africa,1977,5.385864e+11
6,Africa,1982,6.998437e+11
7,Africa,1987,5.139205e+11
8,Africa,1992,9.990865e+11
9,Africa,1997,1.035296e+12


# **3. Data Cleaning & Preparation**
Data Cleaning & Preparation là quá trình chuyển dữ liệu thô thành dữ liệu sẵn sàng cho phân tích và mô hình hóa. Nội dung thường bao gồm:


*   Nhận diện và xử lý giá trị thiếu (NULL/NaN/blank/sentinel)
*   Chuẩn hóa kiểu dữ liệu (number, datetime, boolean, category)


*   Chuẩn hóa chuỗi và biến phân loại (trim, chữ hoa/thường, mapping)
*   Phát hiện và xử lý trùng lặp (trùng dòng, trùng khóa nghiệp vụ)


*   Phát hiện giá trị không hợp lệ và giá trị bất thường (invalid/outlier)
*   Kiểm tra kết quả sau làm sạch và tổng hợp báo cáo thay đổi





## **3.1 Case study: Air Quality (UCI)**

Chúng ta sử dụng dữ liệu thực từ UCI (Air Quality), ghi theo mỗi giờ.S


*   Cột thời gian: Date (DD/MM/YYYY), Time (HH.MM.SS)
*   Biến đo: nồng độ khí (CO, NOx, NO2, Benzene, . . . ) và phản hồi cảm biến(PT08.*), kèm nhiệt độ/độ ẩm.
*   Missing: các giá trị thiếu được gắn bằng -200

Dataset: https://drive.google.com/file/d/1I4ONhf2cdXp7tP4Wc0dBZOuWXYJ6oZDO/view
https://www.kaggle.com/datasets/dakshbhalala/uci-air-quality-dataset/data
https://drive.google.com/file/d/1wGajpW7N2zNZq2Xbg6MqxiIZ1xjOy9mA/view?usp=drive_link

**Chuẩn bị: Import và đọc dataset**

In [44]:
!gdown --id 1wGajpW7N2zNZq2Xbg6MqxiIZ1xjOy9mA

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1wGajpW7N2zNZq2Xbg6MqxiIZ1xjOy9mA
To: /content/AirQualityUCI.csv
100% 752k/752k [00:00<00:00, 115MB/s]


In [45]:
import numpy as pd
import pandas as pd

DATA_PATH = '/content/AirQualityUCI.csv'
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(9357, 15)


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,03/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,03/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,03/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,03/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,03/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


**Bước 1: Profiling**

Mục tiêu là **xem nhanh dữ liệu** và đếm **mức độ thiếu** ở các cột quan trọng. Với Air Quality, dữ liệu thiếu thường được mã hóa bằng -200.

In [46]:
use_cols = ["CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)", "PT08.S2(NMHC)",
            "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH"]

(df[use_cols] == -200).sum()

,0
CO(GT),1683
PT08.S1(CO),366
NMHC(GT),8443
C6H6(GT),366
PT08.S2(NMHC),366
NOx(GT),1639
PT08.S3(NOx),366
NO2(GT),1642
PT08.S4(NO2),366
PT08.S5(O3),366


**Bước 2: Tạo cột thời gian (timestamp)**

In [54]:
df["ts"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors ="coerce"
)

df[["Date", "Time", "ts"]].head()

,Date,Time,ts
0,03/10/2004,18:00:00,2004-10-03 18:00:00
1,03/10/2004,19:00:00,2004-10-03 19:00:00
2,03/10/2004,20:00:00,2004-10-03 20:00:00
3,03/10/2004,21:00:00,2004-10-03 21:00:00
4,03/10/2004,22:00:00,2004-10-03 22:00:00


**Bước 3: Chuẩn hóa missing (-200 → NaN/NULL)**

Mục tiêu là đưa missing về dạng chuẩn để dễ xử lý. Ở đây, ta thay -200 bằng NaN/NULL.

In [55]:
import numpy as np

df[use_cols] = df[use_cols].replace(-200, np.nan)
df[use_cols].isna().sum()

,0
CO(GT),1683
PT08.S1(CO),366
NMHC(GT),8443
C6H6(GT),366
PT08.S2(NMHC),366
NOx(GT),1639
PT08.S3(NOx),366
NO2(GT),1642
PT08.S4(NO2),366
PT08.S5(O3),366


In [56]:
df

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,ts
0,03/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,2004-10-03 18:00:00
1,03/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,2004-10-03 19:00:00
2,03/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,2004-10-03 20:00:00
3,03/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,2004-10-03 21:00:00
4,03/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,2004-10-03 22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9352,04/04/2005,10:00:00,3.1,1314.0,NaN,13.5,1101.0,472.0,539.0,190.0,1374.0,1729.0,21.9,29.3,0.7568,2005-04-04 10:00:00
9353,04/04/2005,11:00:00,2.4,1163.0,NaN,11.4,1027.0,353.0,604.0,179.0,1264.0,1269.0,24.3,23.7,0.7119,2005-04-04 11:00:00
9354,04/04/2005,12:00:00,2.4,1142.0,NaN,12.4,1063.0,293.0,603.0,175.0,1241.0,1092.0,26.9,18.3,0.6406,2005-04-04 12:00:00
9355,04/04/2005,13:00:00,2.1,1003.0,NaN,9.5,961.0,235.0,702.0,156.0,1041.0,770.0,28.3,13.5,0.5139,2005-04-04 13:00:00


**Bước 4: Lọc dữ liệu cột ts**

Mục tiêu là đảm bảo dữ liệu có thứ tự thời gian rõ ràng. Các dòng không parse được thời gian sẽ bị loại bỏ.

In [57]:
df = df.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True)
df[["ts"] + use_cols].head()

,ts,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH
0,2004-01-04 00:00:00,1.6,1143.0,106.0,6.3,825.0,96.0,986.0,86.0,1477.0,978.0,12.0,61.6
1,2004-01-04 01:00:00,1.2,1044.0,100.0,5.1,770.0,85.0,1031.0,70.0,1425.0,944.0,11.5,63.9
2,2004-01-04 02:00:00,1.1,1034.0,71.0,4.1,716.0,50.0,1085.0,55.0,1405.0,891.0,10.7,67.2
3,2004-01-04 03:00:00,0.9,956.0,72.0,4.0,713.0,NaN,1099.0,NaN,1422.0,849.0,9.0,73.1
4,2004-01-04 04:00:00,0.7,909.0,44.0,2.4,615.0,57.0,1237.0,49.0,1322.0,790.0,10.2,66.6


**Bước 5: Điền missing đơn giản (impute)**

Mục tiêu là tạo phiên bản dữ liệu ít thiếu hơn để phục vụ phân tích. Trong Pandas, ta dùng **forward-fill**. Trong SQL, ta minh họa bằng điền trung bình để giữ câu lệnh đơn giản

In [58]:
for c in use_cols:
    df[c + "_filled"] = df[c].ffill()

df[["ts"] + [c + "_filled" for c in use_cols]].head()

,ts,CO(GT)_filled,PT08.S1(CO)_filled,NMHC(GT)_filled,C6H6(GT)_filled,PT08.S2(NMHC)_filled,NOx(GT)_filled,PT08.S3(NOx)_filled,NO2(GT)_filled,PT08.S4(NO2)_filled,PT08.S5(O3)_filled,T_filled,RH_filled
0,2004-01-04 00:00:00,1.6,1143.0,106.0,6.3,825.0,96.0,986.0,86.0,1477.0,978.0,12.0,61.6
1,2004-01-04 01:00:00,1.2,1044.0,100.0,5.1,770.0,85.0,1031.0,70.0,1425.0,944.0,11.5,63.9
2,2004-01-04 02:00:00,1.1,1034.0,71.0,4.1,716.0,50.0,1085.0,55.0,1405.0,891.0,10.7,67.2
3,2004-01-04 03:00:00,0.9,956.0,72.0,4.0,713.0,50.0,1099.0,55.0,1422.0,849.0,9.0,73.1
4,2004-01-04 04:00:00,0.7,909.0,44.0,2.4,615.0,57.0,1237.0,49.0,1322.0,790.0,10.2,66.6


**Bước 6: Xác định outlier**

Mục tiêu là đánh dấu các giá trị bất thường để kiểm tra. Ở đây dùng quy tắc đơn giản: gắn cờ nếu CO(GT) vượt một ngưỡng cố định (ví dụ 5).

In [62]:
THRESH = 5.0

df["is_outlier_co"] = (df["CO(GT)_filled"] > THRESH).astype(int)
df[["ts", "CO(GT)_filled", "is_outlier_co"]].head(20)

,ts,CO(GT)_filled,is_outlier_co
0,2004-01-04 00:00:00,1.6,0
1,2004-01-04 01:00:00,1.2,0
2,2004-01-04 02:00:00,1.1,0
3,2004-01-04 03:00:00,0.9,0
4,2004-01-04 04:00:00,0.7,0
5,2004-01-04 05:00:00,0.9,0
6,2004-01-04 06:00:00,1.7,0
7,2004-01-04 07:00:00,4.2,0
8,2004-01-04 08:00:00,6.2,1
9,2004-01-04 09:00:00,4.6,0


# **4. Window Function**

Trong xử lý dữ liệu, window function là một kỹ thuật tạo đặc trưng bằng cách tính toán trên một cửa sổ các dòng liên quan đến dòng hiện tại, nhưng không làm mất từng dòng dữ liệu.

Điểm khác với phép gom nhóm:


*   Gom nhóm (group aggregation): thu gọn dữ liệu, số dòng giảm (ví dụ: trung bình theo nhóm).
*   Window (analytic): giữ nguyên số dòng, nhưng tạo thêm cột kết quả chạy theo nhóm và theo thứ tự (ví dụ: giá trị trước đó, trung bình trượt, thứ hạng).

Vì giữ nguyên từng dòng, window functions đặc biệt hữu ích cho dữ liệu theo thời gian và
dữ liệu cần so sánh giữa các dòng liền kề.


## **4.1 Case study: Stocks (Vega Datasets)**

Dataset Stocks có cấu trúc gọn nên phù hợp để minh hoạ window functions.


*   Cột nhóm: symbol
*   Cột thời gian: date
*   Giá trị số: price

Mục tiêu: minh hoạ row number, lag/delta, moving average, running sum, và top-k theo năm bằng cả Pandas và SQL

Các bạn có thể tải dataset ở đây: https://drive.google.com/file/d/1T2NoXq0obThoiSFp5aNI2eAJ1B_HPbqL/view

**Download library and import dataset**

In [63]:
!gdown --id 1T2NoXq0obThoiSFp5aNI2eAJ1B_HPbqL

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1T2NoXq0obThoiSFp5aNI2eAJ1B_HPbqL
To: /content/stocks.csv
100% 12.2k/12.2k [00:00<00:00, 34.2MB/s]


In [64]:
import pandas as pd

df = pd.read_csv('/content/stocks.csv')
df

,symbol,date,price
0,MSFT,Jan 1 2000,39.81
1,MSFT,Feb 1 2000,36.35
2,MSFT,Mar 1 2000,43.22
3,MSFT,Apr 1 2000,28.37
4,MSFT,May 1 2000,25.45
...,...,...,...
555,AAPL,Nov 1 2009,199.91
556,AAPL,Dec 1 2009,210.73
557,AAPL,Jan 1 2010,192.06
558,AAPL,Feb 1 2010,204.62


**Convert date to YYYY-MM-DD**

In [65]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%b %d %Y",
    errors = "coerce"
)

df["date"] = df["date"].dt.strftime("%Y-%m-%d")
df

,symbol,date,price
0,MSFT,2000-01-01,39.81
1,MSFT,2000-02-01,36.35
2,MSFT,2000-03-01,43.22
3,MSFT,2000-04-01,28.37
4,MSFT,2000-05-01,25.45
...,...,...,...
555,AAPL,2009-11-01,199.91
556,AAPL,2009-12-01,210.73
557,AAPL,2010-01-01,192.06
558,AAPL,2010-02-01,204.62


**Bước 1: Row number theo symbol**

Mục tiêu là đánh số thứ tự theo thời gian trong mỗi symbol.

In [68]:
dfw = df.sort_values(["symbol", "date"]).copy()
dfw["row_num"] = dfw.groupby("symbol").cumcount() + 1 # cumcount()+1: đếm số dòng trong mỗi nhóm theo thứ tự hiện tại (bắt đầu từ 0), nên cộng 1 để ra 1,2,3,...
dfw

,symbol,date,price,row_num
437,AAPL,2000-01-01,25.94,1
438,AAPL,2000-02-01,28.66,2
439,AAPL,2000-03-01,33.95,3
440,AAPL,2000-04-01,31.01,4
441,AAPL,2000-05-01,21.00,5
...,...,...,...,...
118,MSFT,2009-11-01,29.27,119
119,MSFT,2009-12-01,30.34,120
120,MSFT,2010-01-01,28.05,121
121,MSFT,2010-02-01,28.67,122


**Bước 2: Lag và delta**

Mục tiêu là lấy giá kỳ trước và tính chênh lệch theo kỳ trong mỗi symbol.

In [70]:
dfw["prev_price"] = dfw.groupby("symbol")["price"].shift(1)

In [71]:
dfw["delta"] = dfw["price"] - dfw["prev_price"]
dfw

,symbol,date,price,row_num,prev_price,delta
437,AAPL,2000-01-01,25.94,1,NaN,NaN
438,AAPL,2000-02-01,28.66,2,25.94,2.72
439,AAPL,2000-03-01,33.95,3,28.66,5.29
440,AAPL,2000-04-01,31.01,4,33.95,-2.94
441,AAPL,2000-05-01,21.00,5,31.01,-10.01
...,...,...,...,...,...,...
118,MSFT,2009-11-01,29.27,119,27.48,1.79
119,MSFT,2009-12-01,30.34,120,29.27,1.07
120,MSFT,2010-01-01,28.05,121,30.34,-2.29
121,MSFT,2010-02-01,28.67,122,28.05,0.62


**Bước 3: Moving average và running sum**

Mục tiêu là tạo trung bình trượt 3 k và tổng cộng dồn trong mỗi symbol.

In [72]:
dfw["ma3"] = (
    dfw.groupby("symbol")["price"]
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(level = 0, drop=True)
)

dfw["running_sum"] = (
    dfw.groupby("symbol")["price"]
    .cumsum()
)

In [74]:
dfw

,symbol,date,price,row_num,prev_price,delta,ma3,running_sum
437,AAPL,2000-01-01,25.94,1,NaN,NaN,25.940000,25.94
438,AAPL,2000-02-01,28.66,2,25.94,2.72,27.300000,54.60
439,AAPL,2000-03-01,33.95,3,28.66,5.29,29.516667,88.55
440,AAPL,2000-04-01,31.01,4,33.95,-2.94,31.206667,119.56
441,AAPL,2000-05-01,21.00,5,31.01,-10.01,28.653333,140.56
...,...,...,...,...,...,...,...,...
118,MSFT,2009-11-01,29.27,119,27.48,1.79,27.413333,2926.76
119,MSFT,2009-12-01,30.34,120,29.27,1.07,29.030000,2957.10
120,MSFT,2010-01-01,28.05,121,30.34,-2.29,29.220000,2985.15
121,MSFT,2010-02-01,28.67,122,28.05,0.62,29.020000,3013.82


**Bước 4: Top-3 (RANK)**

Mục tiêu là xếp hạng theo năm trong từng symbol và lấy top-3 tháng có giá cao nhất

In [76]:
dfw["date"] = pd.to_datetime(dfw["date"])
dfw ["year"] = dfw["date"].dt.year
dfw ["rank_year"] = dfw.groupby(["symbol", "year"])["price"].rank(
    ascending=False, method="min"
)

top3 = dfw[dfw["rank_year"] <= 3].sort_values(["symbol", "year", "rank_year", "date"])

top3

,symbol,date,price,row_num,prev_price,delta,ma3,running_sum,year,rank_year
439,AAPL,2000-03-01,33.95,3,28.66,5.29,29.516667,88.55,2000,1.0
440,AAPL,2000-04-01,31.01,4,33.95,-2.94,31.206667,119.56,2000,2.0
444,AAPL,2000-08-01,30.47,8,25.41,5.06,27.356667,222.63,2000,3.0
452,AAPL,2001-04-01,12.74,16,11.03,1.71,10.963333,304.68,2001,1.0
454,AAPL,2001-06-01,11.62,18,9.98,1.64,11.446667,326.28,2001,2.0
...,...,...,...,...,...,...,...,...,...,...
118,MSFT,2009-11-01,29.27,119,27.48,1.79,27.413333,2926.76,2009,2.0
117,MSFT,2009-10-01,27.48,118,25.49,1.99,25.800000,2897.49,2009,3.0
122,MSFT,2010-03-01,28.80,123,28.67,0.13,28.506667,3042.62,2010,1.0
121,MSFT,2010-02-01,28.67,122,28.05,0.62,29.020000,3013.82,2010,2.0
